In [43]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

import pandas as pd
from src.utils.db import get_connection

import ast
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

# 1. 데이터 불러오기
conn = get_connection()

reviews_df = pd.read_sql(
    "SELECT * FROM steam_indie_reviews",
    conn
)

metadata_df = pd.read_sql(
    "SELECT * FROM steam_stratified_sample",
    conn
)

conn.close()


reviews_df['author_playtime_forever'] = reviews_df['author_playtime_forever'].astype('int64')
reviews_df['author_playtime_at_review'] = reviews_df['author_playtime_at_review'].astype('int64')
metadata_df['owners_lower'] = metadata_df['owners_lower'].astype('int64')
metadata_df['positive'] = metadata_df['positive'].astype('int64')
metadata_df['negative'] = metadata_df['negative'].astype('int64')
metadata_df['total_reviews'] = metadata_df['total_reviews'].astype('int64')


In [ ]:
merged_df = pd.merge(reviews_df, metadata_df[['appid', 'genres', 'stratum', 'owners_lower']], on='appid')
merged_df['playtime_after_review_hrs'] = (merged_df['author_playtime_forever'] - merged_df['author_playtime_at_review']) / 60
merged_df = merged_df[merged_df['playtime_after_review_hrs'] >= 0]

# 3. 장르 조합 분석용 전처리 (Indie 제외)
def get_genre_combo_no_indie(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        filtered = [g for g in genres if g != 'Indie']
        return ", ".join(sorted(filtered)) if filtered else None
    except:
        return None

metadata_df['genre_combo_clean'] = metadata_df['genres'].apply(get_genre_combo_no_indie)
df_genre_market = metadata_df.dropna(subset=['genre_combo_clean'])
df_genre_market = df_genre_market[df_genre_market['stratum'].isin(['large_high', 'mid_high', 'small_high'])]

print("데이터 로드 및 전처리 완료")

In [22]:
df_genre_market['stratum'].value_counts()

stratum
large_high    29
mid_high      24
small_high    20
Name: count, dtype: int64

In [23]:
# 1. 상위 15개 장르 조합 추출
top_combos = df_genre_market['genre_combo_clean'].value_counts().nlargest(15).index
df_top_combos = df_genre_market[df_genre_market['genre_combo_clean'].isin(top_combos)].copy()

# 2. 중앙값 기준으로 정렬 순서 계산
combo_order = df_top_combos.groupby('genre_combo_clean')['owners_lower'].median().sort_values(ascending=True).index.tolist()

# 3. Plotly Box Plot 생성
fig_market = px.box(
    df_top_combos, 
    y='genre_combo_clean', 
    x='owners_lower', 
    color='genre_combo_clean',
    points=False, # 이상치 점 표시 제외 (박스에 집중)
    category_orders={'genre_combo_clean': combo_order},
    title='장르 조합별 Owners 분포 (Indie 제외, 상위 15개)',
    labels={'owners_lower': '판매량 하한선 (Log Scale)', 'genre_combo_clean': '장르 조합'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)

# 4. 레이아웃 설정 (로그 스케일 및 스타일)
fig_market.update_layout(
    xaxis_type="log", # 판매량 격차를 위해 로그 스케일 적용
    showlegend=False,
    template="plotly_white",
    height=800,
    margin=dict(l=200) # 장르 조합명이 길 경우를 대비해 왼쪽 여백 확보
)

# 축 눈금 표시 개선 (10k, 100k 등)
fig_market.update_xaxes(dtick=1) 

fig_market.show()

In [24]:
# 1. 게임별 중앙값 플레이타임 계산
game_stats = reviews_df.groupby('appid')['author_playtime_forever'].median().reset_index()
game_stats['median_playtime_hrs'] = game_stats['author_playtime_forever'] / 60

# 2. 메타데이터와 결합 (판매량 정보 가져오기)
df_scatter = pd.merge(game_stats, metadata_df[['appid', 'genres', 'name_store', 'owners_lower', 'stratum']], on='appid')

In [25]:
# 1. Stratum별 중앙값 집계
stratum_summary = df_scatter.groupby('stratum').agg({
    'owners_lower': 'median',
    'median_playtime_hrs': 'median'
}).reset_index()

# 2. 막대 그래프 시각화 (판매량과 플레이타임 비교)
fig_stratum_market = px.bar(
    stratum_summary, 
    x='stratum', 
    y='median_playtime_hrs',
    color='stratum',
    text_auto='.1f',
    title='Stratum별 평균(중앙값) 플레이타임 비교',
    labels={'median_playtime_hrs': '평균 플레이타임 (시간)', 'stratum': '그룹'},
    category_orders={'stratum': ['large_high', 'mid_high', 'small_high']}
)

fig_stratum_market.update_layout(template="plotly_white", showlegend=False)
fig_stratum_market.show()

In [26]:
# 1. 장르별로 데이터를 쪼개고 판매량 구간별 플레이타임 중앙값 계산
target_genres = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]

# 장르 확장 및 필터링
df_genre_trend = df_scatter.copy()
df_genre_trend['genres_list'] = df_genre_trend['genres'].apply(ast.literal_eval)
df_genre_trend = df_genre_trend.explode('genres_list')
df_genre_trend = df_genre_trend[df_genre_trend['genres_list'].isin(target_genres)]
df_genre_trend = df_genre_trend[df_genre_trend['stratum'].isin(['large_high', 'mid_high', 'small_high'])]

# 장르 & 판매량 구간별 집계
genre_market_trend = df_genre_trend.groupby(['genres_list', 'owners_lower'])['median_playtime_hrs'].median().reset_index()

# 2. 선 그래프 시각화 (Trend Line)
fig_trend = px.line(
    genre_market_trend, 
    x='owners_lower', 
    y='median_playtime_hrs', 
    color='genres_list',
    markers=True,
    log_x=True, # 판매량은 로그 스케일이 보기 편함
    title='장르별 판매량 증가에 따른 플레이타임 변화 추이',
    labels={'owners_lower': '판매량 구간', 'median_playtime_hrs': '중앙값 플레이타임 (h)', 'genres_list': '장르'}
)

fig_trend.update_layout(template="plotly_white", hovermode="x unified")
fig_trend.show()

In [27]:
import pandas as pd
from scipy import stats

# 1. 게임별 중앙값 플레이타임 준비 (앞선 셀의 결과 활용)
game_stats = reviews_df.groupby('appid')['author_playtime_forever'].median().reset_index()
game_stats['median_playtime_hrs'] = game_stats['author_playtime_forever'] / 60

# 2. 판매량 데이터와 결합
df_corr_analysis = pd.merge(game_stats, metadata_df[['appid', 'genres', 'owners_lower', 'stratum']], on='appid')

# 3. 전체 스피어먼 상관계수 계산
overall_corr = df_corr_analysis['median_playtime_hrs'].corr(df_corr_analysis['owners_lower'], method='spearman')

print(f"전체 게임의 플레이타임-판매량 상관계수 (Spearman): {overall_corr:.4f}")

# 4. 장르별 상관계수 계산 (8대 타겟 장르 대상)
target_genres = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]

# 장르 데이터 확장
df_corr_analysis['genres_list'] = df_corr_analysis['genres'].apply(ast.literal_eval)
df_genre_corr = df_corr_analysis.explode('genres_list')
df_genre_corr = df_genre_corr[df_genre_corr['genres_list'].isin(target_genres)]

# 장르별로 그룹화하여 상관계수 추출
genre_specific_corr = df_genre_corr.groupby('genres_list').apply(
    lambda x: x['median_playtime_hrs'].corr(x['owners_lower'], method='spearman')
).reset_index(name='spearman_corr')

# 결과 출력
print("\n--- 장르별 상관계수 순위 ---")
display(genre_specific_corr.sort_values(by='spearman_corr', ascending=False))

전체 게임의 플레이타임-판매량 상관계수 (Spearman): 0.3713

--- 장르별 상관계수 순위 ---


,genres_list,spearman_corr
7,Strategy,0.626330
2,Casual,0.417402
3,RPG,0.346913
1,Adventure,0.310822
5,Simulation,0.249546
0,Action,0.211743
6,Sports,-0.866025
4,Racing,-1.000000


In [39]:
import pandas as pd
import ast
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

# 1. 데이터 준비
target_genres = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]

# 게임별 지표 계산 및 결합
game_market = pd.merge(
    reviews_df.groupby('appid')['author_playtime_forever'].median().reset_index(),
    metadata_df[['appid', 'genres', 'owners_lower', 'positive', 'total_reviews']], 
    on='appid'
)
game_market['median_playtime_hrs'] = game_market['author_playtime_forever'] / 60

# 로그 스케일 적용을 위해 0을 1로 치환 (0은 로그 계산이 안 됨)
game_market['owners_plot'] = game_market['owners_lower'].replace(0, 1)

# 장르 확장 및 필터링
game_market['genres_list'] = game_market['genres'].apply(ast.literal_eval)
df_plot = game_market.explode('genres_list')
df_plot = df_plot[df_plot['genres_list'].isin(target_genres)]

In [40]:
# 구간별 데이터 집계
trend_data = df_plot.groupby(['genres_list', 'owners_plot']).agg({
    'median_playtime_hrs': 'median',
    'positive': 'sum',
    'total_reviews': 'sum'
}).reset_index()
trend_data['positive_ratio'] = (trend_data['positive'] / trend_data['total_reviews']) * 100

In [41]:
import numpy as np

# 1. 분석 대상 장르 및 데이터 준비
target_genres = ["Action", "RPG", "Strategy", "Simulation", "Adventure", "Casual", "Sports", "Racing"]
n_genres = len(target_genres)

# 데이터 집계
# owners_plot이 0인 경우를 대비해 1로 치환하여 로그 스케일 대응
trend_data['owners_plot'] = trend_data['owners_plot'].replace(0, 1)

In [42]:
# 2. 서브플롯 생성 (8행 2열)
# 행: 장르별 / 열: 플레이타임(좌), 긍정률(우)
fig = make_subplots(
    rows=n_genres, cols=2,
    shared_xaxes=True,
    horizontal_spacing=0.1,
    vertical_spacing=0.04,
    subplot_titles=[f"{g} - 몰입도" if i%2==0 else f"{g} - 만족도" 
                    for g in target_genres for i in range(2)]
)

# 3. 장르별 데이터 채우기
for i, genre in enumerate(target_genres):
    row = i + 1
    genre_df = trend_data[trend_data['genres_list'] == genre].sort_values('owners_plot')
    
    # 왼쪽 컬럼: 플레이타임 (Line + Markers)
    fig.add_trace(
        go.Scatter(x=genre_df['owners_plot'], y=genre_df['median_playtime_hrs'],
                mode='lines+markers', name=f"{genre} Playtime",
                line=dict(color='#636EFA', width=2),
                marker=dict(size=6), showlegend=False),
        row=row, col=1
    )
    
    # 오른쪽 컬럼: 긍정률 (Area Chart - 채우기 효과)
    fig.add_trace(
        go.Scatter(x=genre_df['owners_plot'], y=genre_df['positive_ratio'],
                mode='lines+markers', name=f"{genre} Sentiment",
                fill='tozeroy', # 바닥까지 색 채우기
                line=dict(color='#EF553B', width=2),
                marker=dict(size=6), showlegend=False),
        row=row, col=2
    )

# 4. 축 및 레이아웃 세부 설정
fig.update_xaxes(type="log", title_text="판매량 (Log)", row=n_genres, col=1)
fig.update_xaxes(type="log", title_text="판매량 (Log)", row=n_genres, col=2)

# Y축 레이블 및 범위 설정
for i in range(1, n_genres + 1):
    fig.update_yaxes(title_text="h", row=i, col=1)
    fig.update_yaxes(title_text="%", range=[0, 110], row=i, col=2)

fig.update_layout(
    height=2000, # 장르가 많으므로 높이를 충분히 확보
    width=1000,
    title_text="장르별 시장 성과에 따른 몰입도 및 만족도 상세 분석",
    template="plotly_white",
    margin=dict(t=100, b=100)
)

# 서브플롯 제목 폰트 크기 조정
fig.update_annotations(font_size=12)

fig.show()